In [1]:
import re,string,requests
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer

In [13]:
def retrieve_docs_and_clean():
    r = requests.get('https://bola.kompas.com/')

    soup = BeautifulSoup(r.content, 'html.parser')

    link = []
    for i in soup.find('div', {'class':'most__wrap'}).find_all('a'):
        i['href'] = i['href'] + '?page=all'
        link.append(i['href'])

    print(f'Number of links is ({len(link)})')
    print(f'second link is \n')
    print(link[1])
    print('===============================')
    documents = []
    for i in link:
        r = requests.get(i)
        soup = BeautifulSoup(r.content, 'html.parser')

        sen = []
        content_div = soup.find('div', {'class':'read__content'})
        if content_div:
            for p_tag in content_div.find_all('p'):
                sen.append(p_tag.text)
            if sen:
                print(f'number of sentences is {len(sen)} and first sentence is ({sen[0]})')
                documents.append(' '.join(sen))
            else:
                print(f'No sentences found in content_div for link: {i}')
        else:
            print(f"'read__content' div not found for link: {i}")
    print('===============================')
    documents_clean = []
    for d in documents:
        # Correcting the regex pattern to use unicode escape sequence to avoid SyntaxError
        document_test = re.sub(r'[^\u0000-\u007F]+', ' ', d)
        document_test = re.sub(r'@\w+', '', document_test)
        document_test = document_test.lower()
        document_test = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', document_test)
        document_test = re.sub(r'[0-9]', '', document_test)
        document_test = re.sub(r'\s{2,}', ' ', document_test)
        documents_clean.append(document_test)

    print(documents_clean)
    return documents_clean


def get_similar_articles(q, df):
    print("query:", q)
    print("Article with the highest cosine similarity value: ")
    q = [q]
    q_vec = vectorizer.transform(q).toarray().reshape(df.shape[0],)
    print(f'QVec shape is ({q_vec.shape})')
    sim = {}
    # Iterate up to the number of documents (columns in df) to avoid KeyError
    for i in range(df.shape[1]):
        # Ensure the column exists before attempting calculation
        if i in df.columns:
            sim[i] = np.dot(df.loc[:, i].values, q_vec) / np.linalg.norm(df.loc[:, i]) * np.linalg.norm(q_vec)
        else:
            print(f"Warning: Column {i} not found in DataFrame. Skipping.")

    print(sim)
    sim_sorted = sorted(sim.items(), key=lambda x: x[1], reverse=True)

    for k, v in sim_sorted:
        if v != 0.0:
            print("Similarity Value:", v)
            print(docs[k])
            print()

In [14]:
docs = retrieve_docs_and_clean()

Number of links is (10)
second link is 

https://bola.kompas.com/read/2026/07/27/05185038/persija-takluk-0-1-dari-persebaya-shin-tae-yong-soroti-minimnya-waktu-persiapan?page=all
number of sentences is 29 and first sentence is (KOMPAS.com - Duel Timnas Indonesia vs Kamboja dalam Piala AFF atau ASEAN Championship 2026 akan berlangsung pada Senin (27/7/2026) pukul 20.30 WIB. Simak prediksi skor dalam artikel ini.)
number of sentences is 30 and first sentence is (KOMPAS.com - Pelatih Persija Jakarta, Shin Tae-yong, tetap mengapresiasi daya juang dan determinasi anak asuhnya di lapangan hijau meskipun harus menelan kekalahan tipis 0-1 dari Persebaya Surabaya.)
number of sentences is 31 and first sentence is (KOMPAS.com - Laga Timnas Indonesia vs Kamboja dalam Piala AFF atau ASEAN Championship 2026 akan berlangsung pada Senin (27/7/2026) pukul 20.30 WIB. Simak prediksi line-up dalam artikel ini.)
number of sentences is 26 and first sentence is (KOMPAS.com - Pertandingan Timnas Indonesia vs 

In [15]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(docs)
X.shape

(6, 994)

In [16]:
df = pd.DataFrame(X.T.toarray(), index=vectorizer.get_feature_names_out())
print(df.shape)
df.head(20)

(994, 6)


,0,1,2,3,4,5
abdel,0.000000,0.000000,0.040025,0.000000,0.000000,0.000000
absen,0.041291,0.000000,0.000000,0.000000,0.000000,0.000000
ada,0.024496,0.018813,0.000000,0.000000,0.019832,0.017079
adalah,0.042309,0.000000,0.102529,0.093445,0.034254,0.029498
adanya,0.000000,0.000000,0.000000,0.000000,0.033429,0.000000
aff,0.073489,0.000000,0.071235,0.081155,0.000000,0.119551
agar,0.041291,0.000000,0.000000,0.000000,0.000000,0.000000
agung,0.000000,0.000000,0.040025,0.000000,0.000000,0.000000
air,0.000000,0.000000,0.000000,0.045598,0.000000,0.000000
akan,0.091645,0.042231,0.142136,0.141687,0.089036,0.038337


In [17]:
df.tail(20)

,0,1,2,3,4,5
wajah,0.000000,0.000000,0.000000,0.000000,0.000000,0.028788
waktu,0.033859,0.078013,0.000000,0.000000,0.000000,0.000000
walaupun,0.028586,0.000000,0.027710,0.031568,0.000000,0.000000
walsh,0.000000,0.000000,0.040025,0.000000,0.000000,0.000000
warna,0.000000,0.000000,0.000000,0.000000,0.000000,0.028788
warriors,0.041291,0.000000,0.000000,0.000000,0.000000,0.000000
wasit,0.000000,0.031712,0.000000,0.000000,0.000000,0.000000
waspadai,0.041291,0.000000,0.000000,0.000000,0.000000,0.000000
wejangan,0.033859,0.000000,0.032821,0.000000,0.000000,0.000000
wib,0.018329,0.014077,0.017767,0.020241,0.014839,0.012779


In [18]:
q = 'windy'
get_similar_articles(q, df)


query: windy
Article with the highest cosine similarity value: 
QVec shape is ((994,))
{0: np.float64(0.0), 1: np.float64(0.0), 2: np.float64(0.0), 3: np.float64(0.0), 4: np.float64(0.0), 5: np.float64(0.0)}


In [19]:
q = 'adalah'
get_similar_articles(q, df)

query: adalah
Article with the highest cosine similarity value: 
QVec shape is ((994,))
{0: np.float64(0.04230880272453317), 1: np.float64(0.0), 2: np.float64(0.10252912948164647), 3: np.float64(0.0934446372134152), 4: np.float64(0.03425359808603547), 5: np.float64(0.029497706834724177)}
Similarity Value: 0.10252912948164647
kompas com laga timnas indonesia vs kamboja dalam piala aff atau asean championship akan berlangsung pada senin pukul wib simak prediksi line up dalam artikel ini duel grup a asean championship timnas indonesia vs kamboja bakal digelar di stadion pakansari bogor pertandingan tersebut merupakan kans emas bagi skuad garuda untuk meraup tiga poin penuh saat ini posisi timnas indonesia masih tertahan di urutan ketiga klasemen grup a terpaut tiga angka dari vietnam dan singapura yang sudah bertanding lebih dulu menjelang partai timnas indonesia vs kamboja ini catatan sejarah secara telak berpihak kepada kubu tuan rumah baca juga prediksi skor timnas indonesia vs kamboja

In [20]:
q = 'wang'
get_similar_articles(q, df)

query: wang
Article with the highest cosine similarity value: 
QVec shape is ((994,))
{0: np.float64(0.0), 1: np.float64(0.0), 2: np.float64(0.0), 3: np.float64(0.0), 4: np.float64(0.0), 5: np.float64(0.0)}
